# DocFinder — Indexeur Colab (GPU)

Ce notebook indexe vos documents personnels dans Qdrant en tirant profit du GPU Colab.

## Prérequis
1. Qdrant binaire natif tournant sur votre machine locale (port 6333)
2. **ngrok** lancé sur votre machine : `./ngrok http 6333`
3. Vos documents dans un dossier Google Drive (PDF, Word, txt, md)
4. La collection Qdrant initialisée : `python setup_qdrant.py`

## Flux
```
Google Drive → extraction texte → chunking → YAKE keywords
    → embeddings GPU (sentence-transformers) → Qdrant via ngrok
```

## 1. Installation des dépendances

In [ ]:
%%capture
!pip install qdrant-client==1.12.1 sentence-transformers==3.3.1 yake==0.4.8 \
             pymupdf==1.25.1 python-docx==1.1.2 pydantic==2.10.3

## 2. Configuration

Remplacez `NGROK_URL` par l'URL fournie par ngrok sur votre machine locale.

In [ ]:
# ─────────────────────────────────────────────────
# Collez ici l'URL ngrok de votre machine locale
# Exemple : https://abc123def.ngrok-free.app
NGROK_URL = "https://VOTRE-URL.ngrok-free.app"

# Dossier Google Drive contenant vos documents
DRIVE_DOCS_PATH = "/content/drive/MyDrive/docs"

# Nom de la collection Qdrant (doit correspondre à setup_qdrant.py)
COLLECTION_NAME = "docfinder"

# Modèle d'embedding (identique au serveur local)
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

# Taille des chunks (en caractères, ~400 mots pour rester sous 512 tokens)
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

# Batch size pour les embeddings GPU
BATCH_SIZE = 64

print(f"✓ Configuration chargée")
print(f"  Qdrant URL : {NGROK_URL}")
print(f"  Documents  : {DRIVE_DOCS_PATH}")

## 3. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive monté")

## 4. Imports et utilitaires

In [ ]:
import hashlib
import os
import re
import uuid
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import fitz  # pymupdf
import yake
from docx import Document as DocxDocument
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, SparseVector
from sentence_transformers import SentenceTransformer

print("✓ Imports OK")

## 5. Extraction de texte (PDF / Word / txt / md)

In [ ]:
def extract_text_from_pdf(path: str) -> str:
    """Extrait le texte brut d'un PDF via pymupdf."""
    doc = fitz.open(path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n".join(pages)


def extract_text_from_word(path: str) -> str:
    """Extrait le texte d'un fichier .docx via python-docx."""
    doc = DocxDocument(path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    return "\n".join(paragraphs)


def extract_text_from_file(path: str) -> Optional[Tuple[str, str]]:
    """
    Détecte le type et extrait le texte d'un fichier.

    Returns:
        Tuple (texte, doc_type) ou None si le format n'est pas supporté.
    """
    ext = Path(path).suffix.lower()
    try:
        if ext == ".pdf":
            return extract_text_from_pdf(path), "pdf"
        elif ext in (".docx", ".doc"):
            return extract_text_from_word(path), "word"
        elif ext == ".txt":
            return Path(path).read_text(encoding="utf-8", errors="ignore"), "txt"
        elif ext == ".md":
            return Path(path).read_text(encoding="utf-8", errors="ignore"), "md"
        else:
            return None
    except Exception as e:
        print(f"  [WARN] Impossible de lire {path} : {e}")
        return None


print("✓ Fonctions d'extraction définies")

## 6. Chunking du texte

In [ ]:
def clean_text(text: str) -> str:
    """Normalise les espaces et supprime les lignes vides excessives."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> List[str]:
    """
    Découpe le texte en chunks avec overlap.
    Essaie de couper aux fins de phrase pour préserver le sens.

    Args:
        text: Texte nettoyé à découper.
        chunk_size: Taille maximale d'un chunk en caractères.
        overlap: Nombre de caractères de chevauchement entre chunks.

    Returns:
        Liste de chunks non vides.
    """
    if len(text) <= chunk_size:
        return [text] if text.strip() else []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        if end < len(text):
            # Cherche la dernière fin de phrase dans la fenêtre
            last_punct = max(
                text.rfind(".", start, end),
                text.rfind("!", start, end),
                text.rfind("?", start, end),
                text.rfind("\n", start, end),
            )
            if last_punct > start + chunk_size // 2:
                end = last_punct + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap

    return chunks


print("✓ Fonctions de chunking définies")

## 7. Extraction de mots-clés YAKE + construction vecteur sparse

In [ ]:
# Extracteur YAKE instancié une seule fois
_yake_extractor = yake.KeywordExtractor(
    lan="fr",
    n=3,
    dedupLim=0.7,
    top=20,
)


def keyword_to_sparse_index(keyword: str) -> int:
    """Hash un mot-clé en index entier (espace 2^20)."""
    return int(hashlib.md5(keyword.lower().encode()).hexdigest(), 16) % (2 ** 20)


def build_sparse_vector(
    text: str,
) -> Tuple[List[int], List[float], List[str]]:
    """
    Extrait les mots-clés YAKE et construit le vecteur sparse.

    Returns:
        Tuple (indices, valeurs, liste_keywords_string)
    """
    kw_list = _yake_extractor.extract_keywords(text)
    if not kw_list:
        return [], [], []

    indices, values, keywords_str = [], [], []
    seen = set()

    for kw, score in kw_list:
        idx = keyword_to_sparse_index(kw)
        if idx in seen:
            continue
        seen.add(idx)
        importance = 1.0 / (score + 1e-9)
        indices.append(idx)
        values.append(importance)
        keywords_str.append(kw)

    # Normalisation
    if values:
        max_val = max(values)
        values = [v / max_val for v in values]

    # Top 10 mots-clés pour le payload
    sorted_kw = sorted(
        zip(keywords_str, values),
        key=lambda x: x[1],
        reverse=True
    )
    top_keywords = [k for k, _ in sorted_kw[:10]]

    return indices, values, top_keywords


print("✓ Extraction YAKE définie")

## 8. Chargement du modèle d'embedding (GPU)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

model = SentenceTransformer(MODEL_NAME, device=device)
print(f"✓ Modèle '{MODEL_NAME}' chargé sur {device}")

## 9. Connexion à Qdrant via ngrok

In [ ]:
qdrant = QdrantClient(url=NGROK_URL)

# Vérification de la connexion
collections = qdrant.get_collections().collections
names = [c.name for c in collections]
print(f"✓ Connecté à Qdrant ({NGROK_URL})")
print(f"  Collections disponibles : {names}")

if COLLECTION_NAME not in names:
    print(f"⚠ Collection '{COLLECTION_NAME}' introuvable.")
    print("  Lancez d'abord : python setup_qdrant.py")

## 10. Scan des documents

In [ ]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc", ".txt", ".md"}

doc_paths = []
for root, _, files in os.walk(DRIVE_DOCS_PATH):
    for fname in files:
        if Path(fname).suffix.lower() in SUPPORTED_EXTENSIONS:
            doc_paths.append(os.path.join(root, fname))

print(f"✓ {len(doc_paths)} document(s) trouvé(s) dans {DRIVE_DOCS_PATH}")
for p in doc_paths[:10]:
    print(f"  - {p}")
if len(doc_paths) > 10:
    print(f"  … et {len(doc_paths) - 10} autres")

## 11. Indexation (extraction → chunking → embedding → Qdrant)

In [ ]:
def doc_id_from_path(path: str) -> str:
    """Hash stable du chemin comme identifiant de document."""
    return hashlib.sha256(path.encode()).hexdigest()[:16]


def index_documents(paths: List[str], batch_size: int = BATCH_SIZE) -> Dict:
    """
    Pipeline complet : extraction → chunking → YAKE → embeddings GPU → Qdrant.

    Returns:
        Statistiques d'indexation.
    """
    stats = {"docs": 0, "chunks": 0, "errors": 0, "skipped": 0}

    # ── Étape 1 : préparer tous les chunks ──────────────────────────────────
    all_chunks = []  # liste de dicts avec toutes les métadonnées

    for path in paths:
        result = extract_text_from_file(path)
        if result is None:
            stats["skipped"] += 1
            continue

        raw_text, doc_type = result
        text = clean_text(raw_text)
        if not text:
            stats["skipped"] += 1
            continue

        doc_id = doc_id_from_path(path)
        title = Path(path).stem
        chunks = chunk_text(text)

        for i, chunk_text_content in enumerate(chunks):
            chunk_id = f"{doc_id}_{i}"
            sparse_idx, sparse_vals, keywords = build_sparse_vector(chunk_text_content)
            all_chunks.append({
                "chunk_id": chunk_id,
                "content": chunk_text_content,
                "payload": {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "title": title,
                    "path": path,
                    "doc_type": doc_type,
                    "content": chunk_text_content,
                    "keywords": keywords,
                    "chunk_index": i,
                },
                "sparse_indices": sparse_idx,
                "sparse_values": sparse_vals,
            })

        stats["docs"] += 1
        print(f"  [{stats['docs']}/{len(paths)}] {title} → {len(chunks)} chunk(s)")

    if not all_chunks:
        print("Aucun chunk à indexer.")
        return stats

    # ── Étape 2 : embeddings en batch (GPU) ─────────────────────────────────
    print(f"\nGénération de {len(all_chunks)} embeddings (GPU)…")
    texts = [c["content"] for c in all_chunks]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    # ── Étape 3 : construction des PointStruct et upsert par batch ──────────
    print("\nUpsert vers Qdrant…")
    upsert_batch_size = 100  # points par appel REST

    for batch_start in range(0, len(all_chunks), upsert_batch_size):
        batch = all_chunks[batch_start : batch_start + upsert_batch_size]
        batch_embeddings = embeddings[batch_start : batch_start + upsert_batch_size]

        points = []
        for chunk_meta, dense_vec in zip(batch, batch_embeddings):
            vector_dict = {"dense": dense_vec.tolist()}
            if chunk_meta["sparse_indices"]:
                vector_dict["sparse"] = SparseVector(
                    indices=chunk_meta["sparse_indices"],
                    values=chunk_meta["sparse_values"],
                )
            points.append(
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector_dict,
                    payload=chunk_meta["payload"],
                )
            )

        qdrant.upsert(
            collection_name=COLLECTION_NAME,
            points=points,
            wait=True,
        )
        stats["chunks"] += len(points)
        print(f"  Batch {batch_start // upsert_batch_size + 1} : {len(points)} points indexés")

    return stats


# ── Lancement ──────────────────────────────────────────────────────────────
print(f"Début de l'indexation de {len(doc_paths)} document(s)…\n")
stats = index_documents(doc_paths)
print(f"""
✓ Indexation terminée
  Documents traités : {stats['docs']}
  Chunks indexés    : {stats['chunks']}
  Ignorés/erreurs   : {stats['skipped']}
""")

## 12. Vérification — test de recherche rapide

In [ ]:
# Remplacez par un mot-clé présent dans vos documents
TEST_QUERY = "votre requête de test"

test_embedding = model.encode(TEST_QUERY, normalize_embeddings=True).tolist()

from qdrant_client.models import NamedVector
hits = qdrant.search(
    collection_name=COLLECTION_NAME,
    query_vector=NamedVector(name="dense", vector=test_embedding),
    limit=3,
    with_payload=True,
)

print(f"Résultats pour '{TEST_QUERY}' :")
for i, h in enumerate(hits, 1):
    title = h.payload.get("title", "?")
    score = round(h.score, 4)
    excerpt = h.payload.get("content", "")[:150]
    print(f"\n[{i}] {title} (score={score})")
    print(f"    {excerpt}…")